<a href="https://colab.research.google.com/github/giraldoal/datos12024/blob/main/Ejercicio_de_aplicaci%C3%B3n_librer%C3%ADa_PDFQuery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Probreza y condiciones de vida en zonas PDET



En el portal de TerriData se hallan las fichas que caracterizan a las 16 subregiones PDET de Colombia. Para conocer más información sobre los municipios PDET, como las iniciativas, inversiones, etc., puede consultar [aquí](https://centralpdet.renovacionterritorio.gov.co/).

Teniendo en cuenta la información suministrada en cada ficha se requiere estructurar la siguiente información de las 16 zonas PDET:
- Nombre de la región PDET
- Total de la población de la región PDET
- Personas en pobreza multidimensional en 2018
- Tasa de ocupación en 2020
- Tasa de desempleo en 2020
- Población rural
- Población urbana

## Descarga de datos

In [ ]:
#@markdown 2. Se descargan los pdf en un archivo zip "FichasPDET.zip" en el directorio de trabajo actual (`/content`)
#@markdown
url_pdets = "https://drive.google.com/file/d/1vsJzZtcGS-IV1Lma4L1LK-zs5l3GFRlj/view?usp=share_link" #@param {type: "string"}

!gdown --fuzzy {url_pdets} --output FichasPDET.zip


Downloading...
From: https://drive.google.com/uc?id=1vsJzZtcGS-IV1Lma4L1LK-zs5l3GFRlj
To: /content/FichasPDET.zip
100% 22.8M/22.8M [00:00<00:00, 26.8MB/s]


In [ ]:
#@markdown 3. Se extraen los archivos comprimidos y se guardan en la carpeta "./FichasPDET"
import os
dir_fichas = "./FichasPDET"
if not os.path.exists(dir_fichas):
  os.mkdir(dir_fichas)
  !unzip  ./FichasPDET.zip -d {dir_fichas}
else:
  print("Los archivos ya estan disponibles")

Los archivos ya estan disponibles


## Extracción de datos

2. Se guarda un archivo en formato xml para identificar mejor la estrategia de extracción de la información

In [ ]:
import os
!pip install pdfquery
import pdfquery as pq

dir_fichas = "./FichasPDET"
nombre_fichas = [os.path.join(dir_fichas, f) for f in os.listdir(dir_fichas) if f.endswith('.pdf')]

pdf = pq.PDFQuery(nombre_fichas[0])
# Solo se utilizará la primera página
pdf.load(0)
#Convertir el PDF a XML
pdf.tree.write('ejemplo.xml', pretty_print=True)

In [ ]:
pdf.extract([
    ('with_formatter', 'text'), # Extrae el texto de la etiqueta encontrada
    ("zona",'LTTextBoxHorizontal:in_bbox("30, 550, 450, 700.0")'),
    ("población", 'LTTextBoxHorizontal:contains("Población:")'),
    ("pobreza", 'LTTextBoxHorizontal:contains("PERSONAS EN 2018")'),
    ("ocupación", 'LTTextBoxHorizontal:contains("TASA DE OCUPACIÓN EN 2020")'),
    ('desempleo', 'LTTextBoxHorizontal:contains("TASA DE DESEMPLEO EN 2020")'),
    ("población rural", 'LTTextBoxHorizontal:contains("Población rural:")'),
    ('población urbana', 'LTTextBoxHorizontal:contains("Población urbana:")'),
  ])

{'zona': 'SIERRA NEVADA-PERIJÁ-\nZONA BANANERA',
 'población': 'Población:\n1.673.353\nhabitantes',
 'pobreza': 'DE VIDA\n529.727 PERSONAS EN 2018 SE',
 'ocupación': '35% TASA DE OCUPACIÓN EN 2020 33,7% TASA DE OCUPACIÓN EN 2020',
 'desempleo': '15,9% TASA DE DESEMPLEO EN 2020 12,8% TASA DE DESEMPLEO EN 2020',
 'población rural': 'Población rural:\n17,2%',
 'población urbana': 'Población urbana:\n82,8%'}

3. Se extraen los datos solicitados y se guardan en una lista  con el orden de columnas según la especificación en la solicitud:


In [ ]:
import pdfquery as pq

datos_extraidos=[]

for i, fname in enumerate(nombre_fichas):
  pdf = pq.PDFQuery(fname)
  pdf.load(0)
  datos_ficha = pdf.extract([
    ('with_formatter', 'text'), # Extrae el texto de la etiqueta encontrada
    ("zona",'LTTextBoxHorizontal:in_bbox("30, 550, 450, 700.0")'),
    ("población", 'LTTextBoxHorizontal:contains("Población:")'),
    ("pobreza", 'LTTextBoxHorizontal:contains("PERSONAS EN 2018")'),
    ("ocupación", 'LTTextBoxHorizontal:contains("TASA DE OCUPACIÓN EN 2020")'),
    ('desempleo', 'LTTextBoxHorizontal:contains("TASA DE DESEMPLEO EN 2020")'),
    ("población rural", 'LTTextBoxHorizontal:contains("Población rural:")'),
    ('población urbana', 'LTTextBoxHorizontal:contains("Población urbana:")'),
  ])
  datos_extraidos.append(datos_ficha)
  print(i,"zona identificada:", datos_ficha['zona'])

0 zona identificada: SIERRA NEVADA-PERIJÁ-
ZONA BANANERA
1 zona identificada: CATATUMBO
2 zona identificada: SUR DE CÓRDOBA
3 zona identificada: ARAUCA
4 zona identificada: PACÍFICO MEDIO
5 zona identificada: BAJO CAUCA Y
NORDESTE
ANTIOQUEÑO
6 zona identificada: SUR DE BOLÍVAR
7 zona identificada: CHOCÓ
8 zona identificada: URABÁ ANTIOQUEÑO
9 zona identificada: PACÍFICO Y FRONTERA
NARIÑENSE
10 zona identificada: SUR DE TOLIMA
11 zona identificada: MONTES DE MARÍA
12 zona identificada: CUENCA DEL CAGUÁN
Y PIEDEMONTE
CAQUETEÑO
13 zona identificada: ALTO PATÍA Y NORTE
DEL CAUCA
14 zona identificada: PUTUMAYO
15 zona identificada: MACARENA GUAVIARE


## Transformación

In [ ]:
import pandas as pd
datos_transformados=[]
encabezado = ['zona', 'población', "pobreza",
              "ocupación", "desempleo",
              "población rural", "población urbana" ]

for i, ficha in enumerate(datos_extraidos):
  fila = []
  dato = ficha['zona']
  # Elimina los saltos de linea
  dato = dato.replace("\n"," ")
  fila.append(dato)

  dato = ficha["población"]
  # Toma la segunda linea
  dato = dato.split("\n")[1]
  fila.append(dato)

  dato = ficha["pobreza"]
  # Toma la segunda linea
  dato = dato.split("\n")[1]
  # Toma la primera palabra
  dato = dato.split(" ")[0]
  fila.append(dato)

  dato = ficha["ocupación"]
  # Toma la primera palabra
  dato = dato.split(" ")[0]
  fila.append(dato)

  dato = ficha["desempleo"]
  # Toma la primera palabra
  dato = dato.split(" ")[0]
  fila.append(dato)

  dato = ficha["población rural"]
  # Toma la última linea
  dato = dato.split("\n")[-1]
  fila.append(dato)

  dato = ficha["población urbana"]
  # Toma la última linea
  dato = dato.split("\n")[-1]

  fila.append(dato)
  datos_transformados.append(fila)

df = pd.DataFrame(datos_transformados)
df.columns = encabezado
df

,zona,población,pobreza,ocupación,desempleo,población rural,población urbana
0,SIERRA NEVADA-PERIJÁ- ZONA BANANERA,1.673.353,529.727,35%,"15,9%","17,2%","82,8%"
1,CATATUMBO,181.588,107.472,"37,2%","6,5%","68,2%","31,8%"
2,SUR DE CÓRDOBA,274.792,145.809,"31,8%","11,6%",48%,52%
3,ARAUCA,188.330,74.363,"33,9%","13,3%","45,3%","54,7%"
4,PACÍFICO MEDIO,385.062,186.599,"31,8%","20,3%","67,7%",32.3%
5,BAJO CAUCA Y NORDESTE ANTIOQUEÑO,425.875,204.980,33%,"8,9%",58.4%,41.6%
6,SUR DE BOLÍVAR,142.381,69.566,"33,1%","12,3%","45,1%","54,9%"
7,CHOCÓ,228.849,153.549,"27,7%","11,9%","66,7%","33,3%"
8,URABÁ ANTIOQUEÑO,483.077,202.523,"38,2%","10,7%","40,4%","59,6%"
9,PACÍFICO Y FRONTERA NARIÑENSE,467.074,305.195,"23,1%","15,2%","68,3%","31,7%"


In [ ]:
import numpy as np

# Clean 'ocupación' column: remove '%' and replace ',' with '.', then convert to float
df['ocupación'] = df['ocupación'].str.replace('%', '', regex=False).str.replace(',', '.', regex=False).astype(float)

# Calculate the 33rd percentile
percentile_33_ocupacion = np.percentile(df['ocupación'], 33)

print(f"The 33rd percentile of the 'ocupación' column is: {percentile_33_ocupacion:.2f}%")

The 33rd percentile of the 'ocupación' column is: 32.98%


## Carga de datos
Finalmente se guardan los datos en el archivo "zonasPDET.csv":

In [ ]:
df.to_csv("zonasPDET.csv")